#Phase 1 — Data Cleaning & Quality Check

Step 1 — Import Libraries
## 1. Import Required Libraries

We will use Pandas for data loading, inspection, cleaning, and analysis.
NumPy will be used for numerical operations, while Path will help us manage
the project folder and file paths.

In [19]:
import pandas as pd
import numpy as np

from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [20]:
project_path = Path(r"D:\StreamFlix Content Analytics Project")

print("Project Path:")
print(project_path)

Project Path:
D:\StreamFlix Content Analytics Project


Verify the Files

In [21]:
files = list(project_path.iterdir())

for file in files:
    print(file.name)

Cleaned_Data
Phase2_Charts
ratings.csv
reviews.csv
schema_and_sql (2).sql
StreamFlix_Data_Dictionary.docx
StreamFlix_ERD.png
StreamFlix_Project_PRD (2).docx
subscribers.csv
titles.csv
watchlist.csv
watch_history.csv


Load the Six CSV Files

In [22]:
subscribers = pd.read_csv(project_path / "subscribers.csv")
titles = pd.read_csv(project_path / "titles.csv")
watch_history = pd.read_csv(project_path / "watch_history.csv")
ratings = pd.read_csv(project_path / "ratings.csv")
reviews = pd.read_csv(project_path / "reviews.csv")
watchlist = pd.read_csv(project_path / "watchlist.csv")

In [23]:
print("Subscribers    :", subscribers.shape)
print("Titles         :", titles.shape)
print("Watch History  :", watch_history.shape)
print("Ratings        :", ratings.shape)
print("Reviews        :", reviews.shape)
print("Watchlist      :", watchlist.shape)

Subscribers    : (15000, 14)
Titles         : (9000, 21)
Watch History  : (650000, 10)
Ratings        : (130000, 5)
Reviews        : (110000, 7)
Watchlist      : (65000, 5)


#Basic Data Profiling

In [24]:
datasets = {
    "subscribers": subscribers,
    "titles": titles,
    "watch_history": watch_history,
    "ratings": ratings,
    "reviews": reviews,
    "watchlist": watchlist
}

profile = []

for name, df in datasets.items():
    profile.append({
        "Table": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

profile_df = pd.DataFrame(profile)

profile_df

,Table,Rows,Columns
0,subscribers,15000,14
1,titles,9000,21
2,watch_history,650000,10
3,ratings,130000,5
4,reviews,110000,7
5,watchlist,65000,5


Inspect Column Names and Data Types

In [25]:
for name, df in datasets.items():
    print("=" * 80)
    print(name.upper())
    print("=" * 80)
    print(df.dtypes)
    print()

SUBSCRIBERS
subscriber_id         object
signup_date           object
country               object
region                object
age                    int64
gender                object
plan_type             object
monthly_price_usd    float64
household_size         int64
primary_device        object
payment_method        object
tenure_months          int64
is_active               bool
churn_date            object
dtype: object

TITLES
title_id                 object
title_name               object
type                     object
primary_genre            object
country                  object
language                 object
release_year              int64
date_added               object
maturity_rating          object
seasons                   int64
content_duration_min      int64
is_original                bool
license_type             object
director                 object
cast                     object
quality_score           float64
popularity_score        float64
license_cost_usd

#Missing Value Analysis

In [26]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        missing_pct = (missing_count / len(df)) * 100
        
        missing_summary.append({
            "Table": name,
            "Column": column,
            "Missing Count": missing_count,
            "Missing Percentage": missing_pct
        })

missing_df = pd.DataFrame(missing_summary)

missing_df

,Table,Column,Missing Count,Missing Percentage
0,subscribers,subscriber_id,0,0.00
1,subscribers,signup_date,0,0.00
2,subscribers,country,0,0.00
3,subscribers,region,0,0.00
4,subscribers,age,0,0.00
5,subscribers,gender,0,0.00
6,subscribers,plan_type,0,0.00
7,subscribers,monthly_price_usd,0,0.00
8,subscribers,household_size,0,0.00
9,subscribers,primary_device,0,0.00


#Show Only Columns With Missing Values

In [27]:
missing_columns = missing_df[
    missing_df["Missing Count"] > 0
].sort_values(
    by="Missing Percentage",
    ascending=False
)

missing_columns

,Table,Column,Missing Count,Missing Percentage
13,subscribers,churn_date,11199,74.66
32,titles,license_expiry,1966,21.84


#Make the Results Easier to Read

#Quick Overall Summary

In [28]:
missing_by_table = (
    missing_columns
    .groupby("Table")
    .agg(
        Columns_With_Missing=("Column", "count"),
        Total_Missing_Values=("Missing Count", "sum")
    )
    .reset_index()
)

missing_by_table

,Table,Columns_With_Missing,Total_Missing_Values
0,subscribers,1,11199
1,titles,1,1966


#Investigate churn_date

In [29]:
# Compare active status with churn_date availability

churn_check = pd.crosstab(
    subscribers["is_active"],
    subscribers["churn_date"].isna()
)

churn_check

churn_date,False,True
is_active,,
False,3801,0
True,0,11199


#Check for Active Subscribers With a Churn Date

In [30]:
active_with_churn_date = subscribers[
    (subscribers["is_active"] == True) &
    (subscribers["churn_date"].notna())
]

print("Active subscribers with churn date:",
      len(active_with_churn_date))

Active subscribers with churn date: 0


In [31]:
active_with_churn_date.head()

,subscriber_id,signup_date,country,region,age,gender,plan_type,monthly_price_usd,household_size,primary_device,payment_method,tenure_months,is_active,churn_date


#Check Inactive Subscribers Without a Churn Date

In [32]:
inactive_without_churn_date = subscribers[
    (subscribers["is_active"] == False) &
    (subscribers["churn_date"].isna())
]

print("Inactive subscribers without churn date:",
      len(inactive_without_churn_date))

Inactive subscribers without churn date: 0


#Investigate license_expiry

In [33]:
missing_license_expiry = titles[
    titles["license_expiry"].isna()
]

print("Titles with missing license expiry:",
      len(missing_license_expiry))

Titles with missing license expiry: 1966


In [34]:
missing_license_expiry.head()

,title_id,title_name,type,primary_genre,country,language,release_year,date_added,maturity_rating,seasons,content_duration_min,is_original,license_type,director,cast,quality_score,popularity_score,license_cost_usd,license_expiry,total_watch_hours,total_plays
0,TTL200000,Toronto Requiem,TV Show,Comedy,South Korea,Korean,2026,2026-04-05,R,6,2754,True,Original,Ravi Nguyen,"Rosa Ferrari, Aria Cohen, Nina Kim, Felix Santos",68.60,0.19,7965521,NaN,"1,300.80",44
6,TTL200006,Golden Berlin,TV Show,Action,South Korea,Korean,2023,2023-10-24,PG,2,600,True,Original,Mateo Rossi,"Bruno Garcia, Bruno Dubois, Diego Lindqvist, H...",76.70,0.53,2266577,NaN,488.20,67
8,TTL200008,Brilliant Frequency,Movie,Comedy,Mexico,Spanish,2026,2026-05-24,TV-MA,0,106,True,Original,Yuki Ferrari,"Rosa Sharma, Chloe Rossi, Lucia Santos, Liam M...",44.60,0.14,1394200,NaN,52.50,54
10,TTL200010,Mumbai Anatomy,Movie,Action,United States,English,2017,2021-04-27,TV-G,0,139,True,Original,Yara Novak,"Ingrid Romano, Joon Larsen, Bruno Andersson, A...",60.90,0.09,9664247,NaN,71.10,50
14,TTL200014,Falling Garden,Movie,Comedy,United Kingdom,English,2026,2026-02-13,TV-14,0,91,True,Original,Ravi Andersson,"Liam Cohen, Joon Adeyemi, Omar Bianchi, Rosa T...",36.20,0.23,1823771,NaN,38.00,56


In [35]:
missing_license_expiry["license_type"].value_counts(dropna=False)

license_type
Original    1966
Name: count, dtype: int64

#Document the Findings in Your Notebook

## 8.7 Missing Value Findings

The missing-value analysis identified missing values in only two columns:

1. **subscribers.churn_date**
   - 11,199 records (74.66%) are missing.
   - All 11,199 records belong to active subscribers.
   - All 3,801 inactive subscribers have a recorded churn date.
   - There are no active subscribers with a churn date and no inactive
     subscribers without a churn date.
   - Therefore, the missing churn dates are considered valid business cases
     and will not be filled.

2. **titles.license_expiry**
   - 1,966 records (21.84%) are missing.
   - All 1,966 records have `license_type = Original`.
   - This indicates that the missing expiry dates are associated with
     StreamFlix Original content.
   - These values will be retained as missing because there is no evidence
     that they represent data-quality errors.

### Conclusion

Overall, the datasets have very good completeness. Missing values are limited
to two business-related fields and appear to have a logical explanation.
No missing values will be removed or imputed at this stage.

#Duplicate Analysis

## 9. Duplicate Analysis

Duplicate records can lead to incorrect counts, inflated watch hours,
duplicate subscribers, and misleading business KPIs.

In this step, we will:

- Check for completely duplicated rows in every table.
- Check duplicate primary identifiers.
- Specifically investigate duplicate `watch_id` values in `watch_history`.
- Specifically investigate duplicate `subscriber_id` values in `subscribers`.

We will identify duplicates before deciding whether any records should be removed.

#Check Completely Duplicate Rows

In [36]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    
    duplicate_summary.append({
        "Table": name,
        "Duplicate Rows": duplicate_count,
        "Duplicate Percentage": (duplicate_count / len(df)) * 100
    })

duplicate_df = pd.DataFrame(duplicate_summary)

duplicate_df

,Table,Duplicate Rows,Duplicate Percentage
0,subscribers,0,0.00
1,titles,0,0.00
2,watch_history,0,0.00
3,ratings,0,0.00
4,reviews,0,0.00
5,watchlist,0,0.00


Check subscriber_id

In [37]:
duplicate_subscriber_ids = subscribers[
    subscribers["subscriber_id"].duplicated(keep=False)
]

print(
    "Duplicate subscriber IDs:",
    duplicate_subscriber_ids["subscriber_id"].nunique()
)

Duplicate subscriber IDs: 0


In [38]:
duplicate_subscriber_ids.head(10)

,subscriber_id,signup_date,country,region,age,gender,plan_type,monthly_price_usd,household_size,primary_device,payment_method,tenure_months,is_active,churn_date


#Check watch_id

In [39]:
duplicate_watch_ids = watch_history[
    watch_history["watch_id"].duplicated(keep=False)
]

print(
    "Duplicate watch IDs:",
    duplicate_watch_ids["watch_id"].nunique()
)

Duplicate watch IDs: 0


In [40]:
duplicate_watch_ids.head(10)

,watch_id,subscriber_id,title_id,watch_date,device,region,content_duration_min,watch_duration_min,completion_pct,completed


## 9.3 Duplicate Analysis Findings

The duplicate analysis was performed across all six datasets.

### Results

- No completely duplicated rows were found in any of the six tables.
- `subscriber_id` was checked for uniqueness in `subscribers.csv` and no
  duplicate subscriber IDs were found.
- `watch_id` was checked for uniqueness in `watch_history.csv` and no
  duplicate watch IDs were found.

### Conclusion

The datasets do not contain duplicate rows or duplicate primary identifiers
based on the checks performed.

Therefore, no records need to be removed due to duplication at this stage.

## 10. Data Type Validation

Correct data types are essential for reliable analysis.

Incorrect data types can cause problems when calculating KPIs, filtering dates,
grouping data by month, calculating durations, or performing mathematical
operations.

In this step, we will validate:

- Date columns
- Numeric columns
- Boolean columns
- Identifier columns

We will first inspect the current data types and then convert columns where
necessary.

#Identify Date Columns

In [41]:
date_columns = {
    "subscribers": ["signup_date", "churn_date"],
    "titles": ["date_added", "license_expiry"],
    "watch_history": ["watch_date"],
    "ratings": ["rating_date"],
    "reviews": ["review_date"],
    "watchlist": ["added_date"]
}

for table, columns in date_columns.items():
    print("=" * 60)
    print(table.upper())
    
    for column in columns:
        print(f"{column}: {datasets[table][column].dtype}")

SUBSCRIBERS
signup_date: object
churn_date: object
TITLES
date_added: object
license_expiry: object
WATCH_HISTORY
watch_date: object
RATINGS
rating_date: object
REVIEWS
review_date: object
WATCHLIST
added_date: object


#Validate Whether Date Values Can Be Converted

## 10.1 Date Validation

The date columns are currently stored as object/string data types.

We will test whether their values can be successfully interpreted as dates.
Invalid date values will be identified rather than silently ignored.

In [42]:
date_validation = []

for table, columns in date_columns.items():
    df = datasets[table]
    
    for column in columns:
        converted = pd.to_datetime(df[column], errors="coerce")
        
        invalid_count = (
            df[column].notna() & converted.isna()
        ).sum()
        
        date_validation.append({
            "Table": table,
            "Column": column,
            "Invalid Dates": invalid_count
        })

date_validation_df = pd.DataFrame(date_validation)

date_validation_df

,Table,Column,Invalid Dates
0,subscribers,signup_date,0
1,subscribers,churn_date,0
2,titles,date_added,0
3,titles,license_expiry,0
4,watch_history,watch_date,0
5,ratings,rating_date,0
6,reviews,review_date,0
7,watchlist,added_date,0


#Check Numeric Columns

In [43]:
numeric_columns = {
    "subscribers": [
        "age",
        "monthly_price_usd",
        "household_size",
        "tenure_months"
    ],
    
    "titles": [
        "release_year",
        "seasons",
        "content_duration_min",
        "quality_score",
        "popularity_score",
        "license_cost_usd",
        "total_watch_hours",
        "total_plays"
    ],
    
    "watch_history": [
        "watch_id",
        "content_duration_min",
        "watch_duration_min",
        "completion_pct"
    ],
    
    "ratings": [
        "rating_id",
        "rating"
    ],
    
    "reviews": [
        "review_id",
        "helpful_votes"
    ],
    
    "watchlist": [
        "watchlist_id"
    ]
}

In [44]:
numeric_validation = []

for table, columns in numeric_columns.items():
    df = datasets[table]
    
    for column in columns:
        converted = pd.to_numeric(df[column], errors="coerce")
        
        invalid_count = (
            df[column].notna() & converted.isna()
        ).sum()
        
        numeric_validation.append({
            "Table": table,
            "Column": column,
            "Invalid Numeric Values": invalid_count
        })

numeric_validation_df = pd.DataFrame(numeric_validation)

numeric_validation_df

,Table,Column,Invalid Numeric Values
0,subscribers,age,0
1,subscribers,monthly_price_usd,0
2,subscribers,household_size,0
3,subscribers,tenure_months,0
4,titles,release_year,0
5,titles,seasons,0
6,titles,content_duration_min,0
7,titles,quality_score,0
8,titles,popularity_score,0
9,titles,license_cost_usd,0


## 10.3 Data Type Validation Findings

The data type validation identified that several date columns were initially
loaded by Pandas as `object` data types.

However, validation using `pd.to_datetime()` confirmed that all non-null values
in the required date columns can be successfully interpreted as dates, with
zero invalid date values.

The required numeric columns were also validated using `pd.to_numeric()`.
No invalid numeric values were identified.

### Findings

- All tested date values are valid and convertible to datetime format.
- No invalid numeric values were found.
- Date columns currently stored as object/string types will be converted to
  datetime format during the cleaning stage.
- Numeric columns already have valid numeric values and do not require
  corrective conversion based on the validation results.

#Convert Date Columns

In [45]:
# Convert date columns to datetime

subscribers["signup_date"] = pd.to_datetime(subscribers["signup_date"])
subscribers["churn_date"] = pd.to_datetime(subscribers["churn_date"])

titles["date_added"] = pd.to_datetime(titles["date_added"])
titles["license_expiry"] = pd.to_datetime(titles["license_expiry"])

watch_history["watch_date"] = pd.to_datetime(watch_history["watch_date"])

ratings["rating_date"] = pd.to_datetime(ratings["rating_date"])

reviews["review_date"] = pd.to_datetime(reviews["review_date"])

watchlist["added_date"] = pd.to_datetime(watchlist["added_date"])

In [46]:
for name, columns in date_columns.items():
    print("=" * 60)
    print(name.upper())
    
    for column in columns:
        print(f"{column}: {datasets[name][column].dtype}")

SUBSCRIBERS
signup_date: datetime64[ns]
churn_date: datetime64[ns]
TITLES
date_added: datetime64[ns]
license_expiry: datetime64[ns]
WATCH_HISTORY
watch_date: datetime64[ns]
RATINGS
rating_date: datetime64[ns]
REVIEWS
review_date: datetime64[ns]
WATCHLIST
added_date: datetime64[ns]


#Update the Dataset Dictionary

In [47]:
datasets["watch_history"].dtypes

watch_id                         int64
subscriber_id                   object
title_id                        object
watch_date              datetime64[ns]
device                          object
region                          object
content_duration_min             int64
watch_duration_min             float64
completion_pct                 float64
completed                         bool
dtype: object

#Outlier Analysis

watch_duration_min > content_duration_min

## 11. Outlier Analysis

Viewing duration is an important measure in the StreamFlix dataset.

The project requirements specifically ask us to identify viewing sessions where
the actual `watch_duration_min` is longer than the corresponding
`content_duration_min`.

Such records may represent data-quality issues, repeated viewing behaviour,
measurement inconsistencies, or other anomalies.

In this step, we will:

1. Compare watch duration with content duration.
2. Identify sessions where watch duration exceeds content duration.
3. Calculate how many records are affected.
4. Calculate the percentage of affected sessions.
5. Inspect the most extreme cases.
6. Decide how these records should be handled after investigation.

#Create Duration Difference

In [48]:
watch_history["duration_difference_min"] = (
    watch_history["watch_duration_min"]
    - watch_history["content_duration_min"]
)

watch_history[
    ["watch_id", "content_duration_min",
     "watch_duration_min", "duration_difference_min"]
].head()

,watch_id,content_duration_min,watch_duration_min,duration_difference_min
0,1,888,826.70,-61.30
1,2,2340,"1,866.40",-473.60
2,3,656,75.00,-581.00
3,4,500,493.20,-6.80
4,5,141,94.60,-46.40


#Identify Potential Outliers

## 11.1 Sessions Exceeding Content Duration

A viewing session will be flagged as a potential duration outlier when:

`watch_duration_min > content_duration_min`

These records will be investigated before any cleaning decision is made.

In [49]:
duration_outliers = watch_history[
    watch_history["watch_duration_min"] >
    watch_history["content_duration_min"]
]

print(
    "Sessions exceeding content duration:",
    len(duration_outliers)
)

Sessions exceeding content duration: 0


#Calculate Percentage

In [50]:
outlier_count = len(duration_outliers)
total_sessions = len(watch_history)

outlier_percentage = (
    outlier_count / total_sessions
) * 100

print(f"Total sessions: {total_sessions:,}")
print(f"Duration outliers: {outlier_count:,}")
print(f"Outlier percentage: {outlier_percentage:.2f}%")

Total sessions: 650,000
Duration outliers: 0
Outlier percentage: 0.00%


#Inspect the Outliers

In [51]:
duration_outliers[
    [
        "watch_id",
        "subscriber_id",
        "title_id",
        "watch_date",
        "device",
        "content_duration_min",
        "watch_duration_min",
        "completion_pct",
        "completed"
    ]
].head(20)

,watch_id,subscriber_id,title_id,watch_date,device,content_duration_min,watch_duration_min,completion_pct,completed


#Find the Most Extreme Cases

## 11.2 Most Extreme Duration Outliers

To understand the severity of the issue, we will examine the sessions with the
largest difference between watch duration and content duration.

In [52]:
duration_outliers[
    [
        "watch_id",
        "title_id",
        "content_duration_min",
        "watch_duration_min",
        "duration_difference_min",
        "completion_pct"
    ]
].sort_values(
    "duration_difference_min",
    ascending=False
).head(20)

,watch_id,title_id,content_duration_min,watch_duration_min,duration_difference_min,completion_pct


## 11.3 Outlier Analysis Findings

The `watch_duration_min` column was compared with `content_duration_min` for
all 650,000 viewing sessions.

A potential duration outlier was defined as a session where:

`watch_duration_min > content_duration_min`

### Results

- Total viewing sessions: 650,000
- Sessions exceeding content duration: 0
- Percentage of duration outliers: 0.00%

### Conclusion

No viewing sessions were found where the watch duration exceeded the
corresponding content duration.

Therefore, no records require correction or removal based on this specific
duration outlier rule.

In [53]:
watch_history.drop(
    columns=["duration_difference_min"],
    inplace=True
)

In [54]:
watch_history.columns

Index(['watch_id', 'subscriber_id', 'title_id', 'watch_date', 'device',
       'region', 'content_duration_min', 'watch_duration_min',
       'completion_pct', 'completed'],
      dtype='object')

#Referential Integrity

subscribers
    │
    │ subscriber_id
    ▼
watch_history
    │
    │ title_id
    ▼
titles

## 12. Referential Integrity

Referential integrity ensures that records in a child table correctly
reference records in their parent tables.

For the StreamFlix project:

- Every `subscriber_id` in `watch_history` should exist in `subscribers`.
- Every `title_id` in `watch_history` should exist in `titles`.

Broken references could lead to incorrect joins, missing business information,
and inaccurate KPIs.

We will identify any unmatched subscriber or title IDs before making any
cleaning decisions.

First check — Subscriber IDs

In [55]:
subscriber_ids_in_history = watch_history["subscriber_id"].isin(
    subscribers["subscriber_id"]
)

print(
    "Watch history records with invalid subscriber_id:",
    (~subscriber_ids_in_history).sum()
)

Watch history records with invalid subscriber_id: 0


Second check — Title IDs

In [56]:
title_ids_in_history = watch_history["title_id"].isin(
    titles["title_id"]
)

print(
    "Watch history records with invalid title_id:",
    (~title_ids_in_history).sum()
)

Watch history records with invalid title_id: 0


Get the actual unmatched IDs

In [57]:
invalid_subscriber_ids = watch_history.loc[
    ~subscriber_ids_in_history,
    "subscriber_id"
].unique()

invalid_title_ids = watch_history.loc[
    ~title_ids_in_history,
    "title_id"
].unique()

print("Unique invalid subscriber IDs:", len(invalid_subscriber_ids))
print("Unique invalid title IDs:", len(invalid_title_ids))

Unique invalid subscriber IDs: 0
Unique invalid title IDs: 0


## 12.1 Referential Integrity Findings

Referential integrity was checked between the `watch_history` table and its
parent tables.

### Results

- All `subscriber_id` values in `watch_history` exist in `subscribers`.
- Invalid subscriber references: **0**
- Unique invalid subscriber IDs: **0**
- All `title_id` values in `watch_history` exist in `titles`.
- Invalid title references: **0**
- Unique invalid title IDs: **0**

### Conclusion

No broken foreign-key relationships were identified.

The `watch_history` table can therefore be reliably joined with the
`subscribers` and `titles` tables using `subscriber_id` and `title_id`.

Specific Data Quality Checks

#Churn Date Validation

## 13. Specific Data Quality Checks

### 13.1 Churn Date Validation

For subscribers who have churned, the `churn_date` should occur after the
`signup_date`.

A churn date that is before or equal to the signup date would indicate a
potential data-quality issue.

We will check only churned subscribers because active subscribers have no
churn date.

In [58]:
# Identify churned subscribers with invalid churn dates

invalid_churn_dates = subscribers[
    (subscribers["is_active"] == False) &
    (subscribers["churn_date"] <= subscribers["signup_date"])
]

print(
    "Churned subscribers with invalid churn dates:",
    len(invalid_churn_dates)
)

Churned subscribers with invalid churn dates: 0


In [59]:
invalid_churn_dates[
    [
        "subscriber_id",
        "signup_date",
        "churn_date",
        "tenure_months",
        "is_active"
    ]
].head(20)

,subscriber_id,signup_date,churn_date,tenure_months,is_active


### 13.1.1 Churn Date Validation Findings

The `churn_date` was compared with `signup_date` for all churned subscribers.

### Result

- Churned subscribers with invalid churn dates: **0**
- No subscriber was found with a `churn_date` before or equal to the
  `signup_date`.

### Conclusion

The churn date relationship is consistent with the signup date for all
churned subscribers. No corrective action is required for this validation.

Active Subscriber Validation

### 13.2 Active Subscriber Churn Date Validation

Active subscribers should not have a recorded `churn_date`.

We will identify any active subscribers where `churn_date` is populated.

In [60]:
active_with_churn_date = subscribers[
    (subscribers["is_active"] == True) &
    (subscribers["churn_date"].notna())
]

print(
    "Active subscribers with a churn date:",
    len(active_with_churn_date)
)

Active subscribers with a churn date: 0


In [61]:
inactive_without_churn_date = subscribers[
    (subscribers["is_active"] == False) &
    (subscribers["churn_date"].isna())
]

print(
    "Inactive subscribers without a churn date:",
    len(inactive_without_churn_date)
)

Inactive subscribers without a churn date: 0


### 13.2.1 Active Subscriber Validation Findings

The relationship between `is_active` and `churn_date` was validated.

### Results

- Active subscribers with a recorded churn date: **0**
- Inactive subscribers without a recorded churn date: **0**

### Conclusion

The `is_active` status and `churn_date` fields are fully consistent.
Missing `churn_date` values for active subscribers represent valid business
cases rather than data-quality errors.

#Completion Percentage Validation

### 13.3 Completion Percentage Validation

The `completion_pct` field represents the percentage of content completed during
a viewing session.

According to the project requirements, it should roughly correspond to:

`(watch_duration_min / content_duration_min) × 100`

We will calculate the expected completion percentage and compare it with the
recorded `completion_pct`.

Because the project uses the word "roughly", small differences caused by
rounding will be considered acceptable.

Calculate Expected Completion %

In [62]:
watch_history["calculated_completion_pct"] = (
    watch_history["watch_duration_min"]
    / watch_history["content_duration_min"]
) * 100

watch_history[
    [
        "watch_duration_min",
        "content_duration_min",
        "completion_pct",
        "calculated_completion_pct"
    ]
].head(10)

,watch_duration_min,content_duration_min,completion_pct,calculated_completion_pct
0,826.70,888,93.10,93.10
1,"1,866.40",2340,79.80,79.76
2,75.00,656,11.40,11.43
3,493.20,500,98.60,98.64
4,94.60,141,67.10,67.09
5,"1,148.00",2860,40.10,40.14
6,78.70,119,66.10,66.13
7,58.60,91,64.30,64.40
8,58.90,93,63.30,63.33
9,61.60,81,76.10,76.05


#### Difference Between Recorded and Calculated Completion %

We will calculate the absolute difference between the recorded completion
percentage and the value derived from viewing and content duration.

In [63]:
watch_history["completion_difference"] = (
    watch_history["completion_pct"]
    - watch_history["calculated_completion_pct"]
).abs()

watch_history[
    [
        "completion_pct",
        "calculated_completion_pct",
        "completion_difference"
    ]
].head(10)

,completion_pct,calculated_completion_pct,completion_difference
0,93.10,93.10,0.00
1,79.80,79.76,0.04
2,11.40,11.43,0.03
3,98.60,98.64,0.04
4,67.10,67.09,0.01
5,40.10,40.14,0.04
6,66.10,66.13,0.03
7,64.30,64.40,0.10
8,63.30,63.33,0.03
9,76.10,76.05,0.05


Define a Reasonable Tolerance

In [64]:
completion_mismatches = watch_history[
    watch_history["completion_difference"] > 1
]

print(
    "Completion percentage mismatches:",
    len(completion_mismatches)
)

Completion percentage mismatches: 0


Calculate the Percentage of Mismatches

In [65]:
completion_mismatch_count = len(completion_mismatches)

completion_mismatch_pct = (
    completion_mismatch_count / len(watch_history)
) * 100

print(f"Total sessions: {len(watch_history):,}")
print(f"Completion mismatches: {completion_mismatch_count:,}")
print(f"Mismatch percentage: {completion_mismatch_pct:.2f}%")

Total sessions: 650,000
Completion mismatches: 0
Mismatch percentage: 0.00%


Inspect the Mismatches

In [66]:
completion_mismatches[
    [
        "watch_id",
        "title_id",
        "content_duration_min",
        "watch_duration_min",
        "completion_pct",
        "calculated_completion_pct",
        "completion_difference",
        "completed"
    ]
].sort_values(
    "completion_difference",
    ascending=False
).head(20)

,watch_id,title_id,content_duration_min,watch_duration_min,completion_pct,calculated_completion_pct,completion_difference,completed


### 13.3.1 Completion Percentage Validation Findings

The recorded `completion_pct` was compared with a calculated completion
percentage using:

`(watch_duration_min / content_duration_min) × 100`

An absolute difference of more than 1 percentage point was considered a
potential mismatch to account for minor rounding differences.

### Results

- Total viewing sessions: **650,000**
- Completion percentage mismatches: **0**
- Mismatch percentage: **0.00%**

### Conclusion

All viewing sessions have a recorded completion percentage that is consistent
with the corresponding watch duration and content duration within the defined
tolerance.

No corrective action is required for `completion_pct`.

#Review Sentiment Validation

### 13.4 Review Sentiment Validation

The `sentiment` column in `reviews` should contain only the following
categories:

- Positive
- Neutral
- Negative

We will inspect the unique sentiment values and identify any unexpected
categories.

In [67]:
# Inspect all sentiment values

reviews["sentiment"].value_counts(dropna=False)

sentiment
Positive    75007
Neutral     22759
Negative    12234
Name: count, dtype: int64

In [68]:
valid_sentiments = {
    "Positive",
    "Neutral",
    "Negative"
}

invalid_sentiments = reviews[
    ~reviews["sentiment"].isin(valid_sentiments)
]

print(
    "Reviews with invalid sentiment:",
    len(invalid_sentiments)
)

Reviews with invalid sentiment: 0


In [69]:
invalid_sentiments[
    ["review_id", "subscriber_id", "title_id", "sentiment"]
].head(20)

,review_id,subscriber_id,title_id,sentiment


### 13.4.1 Review Sentiment Validation Findings

The `sentiment` column was checked against the three valid categories:
`Positive`, `Neutral`, and `Negative`.

### Results

- Positive reviews: **75,007**
- Neutral reviews: **22,759**
- Negative reviews: **12,234**
- Total reviews: **110,000**
- Reviews with invalid sentiment: **0**

### Conclusion

All review records contain a valid sentiment category. No unexpected sentiment
values were identified, so no corrective action is required for the
`sentiment` column.

#Watchlist Referential Integrity

### 13.5 Watchlist Referential Integrity

The `watchlist` table contains titles saved by subscribers.

Every `subscriber_id` in `watchlist` should exist in the `subscribers` table,
and every `title_id` should exist in the `titles` table.

We will identify any watchlist records that reference a subscriber or title
that does not exist in the corresponding parent table.

In [70]:
watchlist_subscriber_check = watchlist["subscriber_id"].isin(
    subscribers["subscriber_id"]
)

print(
    "Watchlist records with invalid subscriber_id:",
    (~watchlist_subscriber_check).sum()
)

Watchlist records with invalid subscriber_id: 0


Code Cell — Title Check

In [71]:
watchlist_title_check = watchlist["title_id"].isin(
    titles["title_id"]
)

print(
    "Watchlist records with invalid title_id:",
    (~watchlist_title_check).sum()
)

Watchlist records with invalid title_id: 0


#Code Cell — Unique Invalid IDs

In [72]:
invalid_watchlist_subscribers = watchlist.loc[
    ~watchlist_subscriber_check,
    "subscriber_id"
].unique()

invalid_watchlist_titles = watchlist.loc[
    ~watchlist_title_check,
    "title_id"
].unique()

print(
    "Unique invalid subscriber IDs:",
    len(invalid_watchlist_subscribers)
)

print(
    "Unique invalid title IDs:",
    len(invalid_watchlist_titles)
)

Unique invalid subscriber IDs: 0
Unique invalid title IDs: 0


### 13.5.1 Watchlist Referential Integrity Findings

Referential integrity was validated for the `watchlist` table.

### Results

- Watchlist records with invalid `subscriber_id`: **0**
- Unique invalid subscriber IDs: **0**
- Watchlist records with invalid `title_id`: **0**
- Unique invalid title IDs: **0**

### Conclusion

All watchlist records correctly reference existing subscribers and titles.
No broken relationships were identified, so no corrective action is required.

#Short-Tenure Active Subscribers

### 13.6 Short-Tenure Active Subscribers

Some subscribers may have a short tenure but still be active.

The project requirements state that this is acceptable and should simply be
noted rather than treated as a data-quality error.

We will first examine the tenure distribution among active subscribers and
identify subscribers with very short tenure.

In [73]:
# Examine tenure distribution for active subscribers

active_subscribers = subscribers[
    subscribers["is_active"] == True
]

active_subscribers["tenure_months"].describe()

count   11,199.00
mean        43.93
std         31.35
min          1.00
25%         17.00
50%         38.00
75%         67.00
max        126.00
Name: tenure_months, dtype: float64

#### Identifying Short-Tenure Active Subscribers

For this quality check, active subscribers with tenure below 3 months will be
flagged for review.

These records are not considered errors. They are only identified so that the
finding can be documented as required by the project.

In [74]:
short_tenure_active = active_subscribers[
    active_subscribers["tenure_months"] < 3
]

print(
    "Active subscribers with tenure below 3 months:",
    len(short_tenure_active)
)

Active subscribers with tenure below 3 months: 486


In [75]:
short_tenure_active[
    [
        "subscriber_id",
        "signup_date",
        "tenure_months",
        "is_active",
        "plan_type",
        "country"
    ]
].head(20)

,subscriber_id,signup_date,tenure_months,is_active,plan_type,country
66,SUB100066,2026-05-18,1,True,Basic with Ads,Japan
87,SUB100087,2026-03-29,2,True,Standard,South Korea
102,SUB100102,2026-04-30,1,True,Standard,India
144,SUB100144,2026-04-10,1,True,Premium,South Korea
170,SUB100170,2026-04-14,1,True,Premium,India
270,SUB100270,2026-05-05,1,True,Premium,United States
277,SUB100277,2026-04-05,1,True,Basic with Ads,South Korea
279,SUB100279,2026-05-31,1,True,Standard,Canada
326,SUB100326,2026-04-30,1,True,Premium,Germany
358,SUB100358,2026-04-24,1,True,Premium,South Korea


### 13.6.1 Short-Tenure Active Subscriber Findings

The tenure distribution of active subscribers was examined.

### Results

- Total active subscribers: **11,199**
- Average tenure: **43.93 months**
- Minimum tenure: **1 month**
- Active subscribers with tenure below 3 months: **486**
- Percentage of active subscribers: **4.34%**

### Conclusion

A total of 486 active subscribers have a tenure of less than 3 months.
These records are not considered data-quality errors because newly subscribed
customers can legitimately remain active despite having a short tenure.

The records will therefore be retained without modification.

## 14. Remove Temporary Analysis Columns

Temporary columns created during the quality checks are not part of the
original dataset.

They will be removed before the final cleaned datasets are prepared.

In [76]:
watch_history.drop(
    columns=[
        "calculated_completion_pct",
        "completion_difference"
    ],
    inplace=True
)

In [77]:
watch_history.shape

(650000, 10)

In [78]:
watch_history.columns.tolist()

['watch_id',
 'subscriber_id',
 'title_id',
 'watch_date',
 'device',
 'region',
 'content_duration_min',
 'watch_duration_min',
 'completion_pct',
 'completed']

Create a Consolidated Data Quality Summary

In [79]:
quality_summary = pd.DataFrame({
    "Check": [
        "Missing values",
        "Duplicate rows",
        "Duplicate subscriber_id",
        "Duplicate watch_id",
        "Invalid dates",
        "Invalid numeric values",
        "Watch duration > content duration",
        "Invalid watch_history subscriber_id",
        "Invalid watch_history title_id",
        "Invalid churn dates",
        "Active subscriber with churn date",
        "Inactive subscriber without churn date",
        "Completion percentage mismatch",
        "Invalid review sentiment",
        "Invalid watchlist subscriber_id",
        "Invalid watchlist title_id"
    ],
    
    "Result": [
        "2 columns affected",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0",
        "0"
    ],
    
    "Status": [
        "Reviewed",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass"
    ]
})

quality_summary

,Check,Result,Status
0,Missing values,2 columns affected,Reviewed
1,Duplicate rows,0,Pass
2,Duplicate subscriber_id,0,Pass
3,Duplicate watch_id,0,Pass
4,Invalid dates,0,Pass
5,Invalid numeric values,0,Pass
6,Watch duration > content duration,0,Pass
7,Invalid watch_history subscriber_id,0,Pass
8,Invalid watch_history title_id,0,Pass
9,Invalid churn dates,0,Pass


## 16. Value Range and Categorical Consistency Checks

In addition to the required Phase 1 checks, we will validate whether important
numeric and categorical fields contain reasonable values.

The checks will include:

- `rating` should be between 1 and 5.
- `completion_pct` should be between 0 and 100.
- Durations should be positive.
- Prices and costs should not be negative.
- `age`, `household_size`, `tenure_months`, and `seasons` should not contain
  negative values.
- Boolean columns should contain valid True/False values.
- Important categorical columns will be inspected for unexpected or
  inconsistent values.

No values will be modified until the validation results are reviewed.

Rating Range

In [80]:
invalid_ratings = ratings[
    (ratings["rating"] < 1) |
    (ratings["rating"] > 5)
]

print("Invalid ratings:", len(invalid_ratings))

Invalid ratings: 0


Completion Percentage Range

In [81]:
invalid_completion = watch_history[
    (watch_history["completion_pct"] < 0) |
    (watch_history["completion_pct"] > 100)
]

print("Invalid completion percentages:", len(invalid_completion))

Invalid completion percentages: 0


Duration Validation

In [82]:
invalid_title_duration = titles[
    titles["content_duration_min"] <= 0
]

invalid_watch_duration = watch_history[
    watch_history["watch_duration_min"] < 0
]

print("Titles with invalid content duration:",
      len(invalid_title_duration))

print("Watch sessions with negative duration:",
      len(invalid_watch_duration))

Titles with invalid content duration: 0
Watch sessions with negative duration: 0


Price and Cost Validation

In [83]:
invalid_monthly_price = subscribers[
    subscribers["monthly_price_usd"] < 0
]

invalid_license_cost = titles[
    titles["license_cost_usd"] < 0
]

print("Subscribers with negative monthly price:",
      len(invalid_monthly_price))

print("Titles with negative license cost:",
      len(invalid_license_cost))

Subscribers with negative monthly price: 0
Titles with negative license cost: 0


Other Numeric Fields

In [84]:
invalid_age = subscribers[
    subscribers["age"] < 0
]

invalid_household = subscribers[
    subscribers["household_size"] < 0
]

invalid_tenure = subscribers[
    subscribers["tenure_months"] < 0
]

invalid_seasons = titles[
    titles["seasons"] < 0
]

print("Negative age:", len(invalid_age))
print("Negative household size:", len(invalid_household))
print("Negative tenure:", len(invalid_tenure))
print("Negative seasons:", len(invalid_seasons))

Negative age: 0
Negative household size: 0
Negative tenure: 0
Negative seasons: 0


Boolean Validation

In [85]:
boolean_columns = {
    "subscribers": ["is_active"],
    "titles": ["is_original"],
    "watch_history": ["completed"],
    "watchlist": ["watched"]
}

for table, columns in boolean_columns.items():
    print("=" * 50)
    print(table.upper())
    
    for column in columns:
        print(column)
        print(datasets[table][column].value_counts(dropna=False))
        print()

SUBSCRIBERS
is_active
is_active
True     11199
False     3801
Name: count, dtype: int64

TITLES
is_original
is_original
False    7034
True     1966
Name: count, dtype: int64

WATCH_HISTORY
completed
completed
False    550637
True      99363
Name: count, dtype: int64

WATCHLIST
watched
watched
False    34952
True     30048
Name: count, dtype: int64



Inspect Important Categorical Values

In [86]:
categorical_columns = {
    "subscribers": [
        "country",
        "region",
        "gender",
        "plan_type",
        "primary_device",
        "payment_method"
    ],
    
    "titles": [
        "type",
        "primary_genre",
        "country",
        "language",
        "maturity_rating",
        "license_type"
    ],
    
    "watch_history": [
        "device",
        "region"
    ],
    
    "reviews": [
        "sentiment"
    ]
}

for table, columns in categorical_columns.items():
    print("=" * 60)
    print(table.upper())
    
    for column in columns:
        print(f"\n{column}:")
        print(datasets[table][column].value_counts(dropna=False).head(20))

SUBSCRIBERS

country:
country
United States     3774
India             1873
United Kingdom    1267
South Korea       1096
Japan              842
Brazil             835
Mexico             749
Spain              671
France             660
Germany            573
Canada             517
Italy              363
Turkey             310
Australia          263
Nigeria            251
Indonesia          228
Argentina          224
Thailand           195
Egypt              166
Sweden             143
Name: count, dtype: int64

region:
region
North America     4291
Europe            3677
East Asia         1938
South Asia        1873
Latin America     1808
Middle East        476
Southeast Asia     423
Oceania            263
Africa             251
Name: count, dtype: int64

gender:
gender
Female         7276
Male           7128
Undisclosed     322
Non-binary      274
Name: count, dtype: int64

plan_type:
plan_type
Standard          6100
Basic with Ads    4458
Premium           4442
Name: count, dtype: in

Create a Compact Range Validation Summary

In [87]:
range_summary = pd.DataFrame({
    "Check": [
        "Rating outside 1-5",
        "Completion % outside 0-100",
        "Content duration <= 0",
        "Watch duration < 0",
        "Negative monthly price",
        "Negative licence cost",
        "Negative age",
        "Negative household size",
        "Negative tenure",
        "Negative seasons"
    ],
    
    "Invalid Records": [
        len(invalid_ratings),
        len(invalid_completion),
        len(invalid_title_duration),
        len(invalid_watch_duration),
        len(invalid_monthly_price),
        len(invalid_license_cost),
        len(invalid_age),
        len(invalid_household),
        len(invalid_tenure),
        len(invalid_seasons)
    ]
})

range_summary

,Check,Invalid Records
0,Rating outside 1-5,0
1,Completion % outside 0-100,0
2,Content duration <= 0,0
3,Watch duration < 0,0
4,Negative monthly price,0
5,Negative licence cost,0
6,Negative age,0
7,Negative household size,0
8,Negative tenure,0
9,Negative seasons,0


### 16.1 Numeric Range Validation Findings

Important numeric fields were checked for invalid or unreasonable values.

### Results

- Ratings outside the 1–5 range: **0**
- Completion percentages outside the 0–100 range: **0**
- Non-positive content durations: **0**
- Negative watch durations: **0**
- Negative monthly subscription prices: **0**
- Negative licence costs: **0**
- Negative ages: **0**
- Negative household sizes: **0**
- Negative tenure values: **0**
- Negative season counts: **0**

### Conclusion

All tested numeric fields fall within the expected ranges. No corrective
action is required based on these range checks.

Boolean values

In [88]:
boolean_columns = {
    "subscribers": ["is_active"],
    "titles": ["is_original"],
    "watch_history": ["completed"],
    "watchlist": ["watched"]
}

for table, columns in boolean_columns.items():
    print("=" * 60)
    print(table.upper())

    for column in columns:
        print(f"\n{column}:")
        print(datasets[table][column].value_counts(dropna=False))

SUBSCRIBERS

is_active:
is_active
True     11199
False     3801
Name: count, dtype: int64
TITLES

is_original:
is_original
False    7034
True     1966
Name: count, dtype: int64
WATCH_HISTORY

completed:
completed
False    550637
True      99363
Name: count, dtype: int64
WATCHLIST

watched:
watched
False    34952
True     30048
Name: count, dtype: int64


Categorical values

In [89]:
categorical_columns = {
    "subscribers": [
        "country",
        "region",
        "gender",
        "plan_type",
        "primary_device",
        "payment_method"
    ],
    
    "titles": [
        "type",
        "primary_genre",
        "country",
        "language",
        "maturity_rating",
        "license_type"
    ],
    
    "watch_history": [
        "device",
        "region"
    ],
    
    "reviews": [
        "sentiment"
    ]
}

for table, columns in categorical_columns.items():
    print("=" * 60)
    print(table.upper())
    
    for column in columns:
        print(f"\n{column}:")
        print(datasets[table][column].value_counts(dropna=False))

SUBSCRIBERS

country:
country
United States     3774
India             1873
United Kingdom    1267
South Korea       1096
Japan              842
Brazil             835
Mexico             749
Spain              671
France             660
Germany            573
Canada             517
Italy              363
Turkey             310
Australia          263
Nigeria            251
Indonesia          228
Argentina          224
Thailand           195
Egypt              166
Sweden             143
Name: count, dtype: int64

region:
region
North America     4291
Europe            3677
East Asia         1938
South Asia        1873
Latin America     1808
Middle East        476
Southeast Asia     423
Oceania            263
Africa             251
Name: count, dtype: int64

gender:
gender
Female         7276
Male           7128
Undisclosed     322
Non-binary      274
Name: count, dtype: int64

plan_type:
plan_type
Standard          6100
Basic with Ads    4458
Premium           4442
Name: count, dtype: in

Final Cleaning and Save

## 18. Final Data Cleaning and Export

All required data-quality checks have been completed.

The validation identified no records requiring deletion or correction.

The only missing values identified were valid business-related missing values:

- subscribers.churn_date is blank for active subscribers.
- titles.license_expiry is blank for titles with license_type = Original.

Date columns were converted to datetime format during the cleaning process.

Temporary columns created during validation will be removed before exporting
the final cleaned datasets.

In [90]:
# Remove temporary analysis columns

temporary_columns = [
    "calculated_completion_pct",
    "completion_difference",
    "duration_difference_min"
]

for column in temporary_columns:
    if column in watch_history.columns:
        watch_history.drop(columns=column, inplace=True)

print("Temporary columns removed successfully.")

Temporary columns removed successfully.


#Create Cleaned Data Folder

## 19. Export Cleaned Datasets

The six cleaned datasets will be saved separately in a dedicated
Cleaned_Data folder.

The original raw CSV files will remain unchanged.

In [91]:
from pathlib import Path

cleaned_path = project_path / "Cleaned_Data"

cleaned_path.mkdir(exist_ok=True)

print("Cleaned data folder:")
print(cleaned_path)

Cleaned data folder:
D:\StreamFlix Content Analytics Project\Cleaned_Data


In [92]:
subscribers.to_csv(
    cleaned_path / "subscribers_clean.csv",
    index=False
)

titles.to_csv(
    cleaned_path / "titles_clean.csv",
    index=False
)

watch_history.to_csv(
    cleaned_path / "watch_history_clean.csv",
    index=False
)

ratings.to_csv(
    cleaned_path / "ratings_clean.csv",
    index=False
)

reviews.to_csv(
    cleaned_path / "reviews_clean.csv",
    index=False
)

watchlist.to_csv(
    cleaned_path / "watchlist_clean.csv",
    index=False
)

print("All 6 cleaned CSV files saved successfully.")

All 6 cleaned CSV files saved successfully.


In [93]:
for file in sorted(cleaned_path.glob("*.csv")):
    print(file.name, "→", file.stat().st_size, "bytes")

ratings_clean.csv → 5218948 bytes
reviews_clean.csv → 10010815 bytes
subscribers_clean.csv → 1533243 bytes
titles_clean.csv → 1917005 bytes
watch_history_clean.csv → 51388849 bytes
watchlist_clean.csv → 2818902 bytes


Final Quality Check

## 20. Final Data Quality Check

A final quality check is performed after cleaning to confirm that:

- All six datasets are available.
- Row counts remain unchanged.
- No duplicate rows were introduced.
- No unexpected missing values were introduced.
- Required date columns remain in datetime format.
- Temporary analysis columns have been removed.

In [94]:
final_quality_check = []

for name, df in datasets.items():
    final_quality_check.append({
        "Table": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate Rows": df.duplicated().sum(),
        "Missing Values": df.isna().sum().sum()
    })

final_quality_df = pd.DataFrame(final_quality_check)

final_quality_df

,Table,Rows,Columns,Duplicate Rows,Missing Values
0,subscribers,15000,14,0,11199
1,titles,9000,21,0,1966
2,watch_history,650000,10,0,0
3,ratings,130000,5,0,0
4,reviews,110000,7,0,0
5,watchlist,65000,5,0,0


#One-Page Data Quality Report

# 21. Data Quality Report

## StreamFlix Content Analytics Project

### Objective

The objective of Phase 1 was to assess the quality, completeness, consistency,
and integrity of the six StreamFlix datasets before performing analysis.

### Dataset Overview

The project contains six datasets:

| Dataset | Rows | Columns |
|---|---:|---:|
| subscribers | 15,000 | 14 |
| titles | 9,000 | 21 |
| watch_history | 650,000 | 10 |
| ratings | 130,000 | 5 |
| reviews | 110,000 | 7 |
| watchlist | 65,000 | 5 |

### Key Data Quality Findings

1. Missing Values

Missing values were found in only two columns:

- subscribers.churn_date: 11,199 missing values (74.66%)
- titles.license_expiry: 1,966 missing values (21.84%)

The missing churn dates are valid because all active subscribers have a blank
churn date, while all inactive subscribers have a recorded churn date.

All missing license_expiry values belong to titles with license_type = Original
and were therefore retained.

2. Duplicate Records

No completely duplicated rows were found in any of the six datasets.

No duplicate subscriber_id values were found in subscribers.

No duplicate watch_id values were found in watch_history.

3. Data Types

All required date fields were successfully converted to datetime format.

All tested numeric fields contained valid numeric values.

4. Outlier Analysis

No viewing sessions were found where watch_duration_min exceeded
content_duration_min.

Total sessions checked: 650,000

Duration outliers: 0

5. Referential Integrity

All subscriber_id values in watch_history exist in subscribers.

All title_id values in watch_history exist in titles.

All subscriber_id values in watchlist exist in subscribers.

All title_id values in watchlist exist in titles.

No broken references were identified.

6. Business Rule Validation

All churned subscribers have a churn_date after signup_date.

No active subscribers have a recorded churn_date.

completion_pct is consistent with watch_duration_min divided by
content_duration_min multiplied by 100, within the defined 1 percentage-point
tolerance.

All review sentiment values are valid: Positive, Neutral, or Negative.

7. Short-Tenure Active Subscribers

486 active subscribers have a tenure of less than 3 months, representing
approximately 4.34% of active subscribers.

These records are considered valid because newly subscribed customers can
legitimately remain active with short tenure.

### Overall Conclusion

The StreamFlix datasets are of high quality and are suitable for downstream
exploratory analysis and KPI development.

No records required deletion based on the Phase 1 quality checks.

The identified missing values have valid business explanations, while
duplicate, range, date, referential-integrity, completion-percentage, and
sentiment checks all passed successfully.

The cleaned datasets have been exported separately while preserving the
original raw data.

Final Notebook Check

In [95]:
print("PHASE 1 DATA CLEANING COMPLETE")
print("=" * 50)

for file in sorted(cleaned_path.glob("*.csv")):
    print("✓", file.name)

print("=" * 50)
print("All cleaned datasets successfully exported.")

PHASE 1 DATA CLEANING COMPLETE
✓ ratings_clean.csv
✓ reviews_clean.csv
✓ subscribers_clean.csv
✓ titles_clean.csv
✓ watch_history_clean.csv
✓ watchlist_clean.csv
All cleaned datasets successfully exported.
